# 04 — Feature engineering y selección de variables
Parte de las tablas Gold y deja dos tablas listas para modelar: `fe_regresion` (amplificación del suelo) y `fe_clasificacion` (detector de onda P), más `fe_columnas_seleccionadas` con las variables que pasaron la selección de cada caso.

Pasos de cada caso:
1. **Variables derivadas:** logaritmos del periodo, del Vs30 y de Sa·R, perfil de suelo de la E.030 y el término de evento (promedio del sismo, que se resta para que el modelo aprenda solo el efecto del suelo).
2. **Multicolinealidad:** se descarta una variable de cada par con |correlación de Pearson| > 0.90.
3. **Selección univariada** (`UnivariateFeatureSelector`, p < 0.05): F-regression para numéricas vs. la amplificación, Chi-cuadrado para categóricas y ANOVA para numéricas vs. la etiqueta de onda P.

La selección se calcula **sin el sismo más reciente**, que luego se usa como *holdout* (30 % de sus estaciones, elegidas con un hash). Las variables de los modelos de daño y de perfiles dependen de la amplificación del suelo, por eso su feature engineering se hace en `07_demanda_sismica`.

## 1. Setup — imports

In [ ]:
import numpy as np
import pandas as pd

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

from pyspark.ml.feature import VectorAssembler, StringIndexer, UnivariateFeatureSelector, QuantileDiscretizer
from pyspark.ml.stat import Correlation

## 2. Catálogo, esquema y parámetros

In [ ]:
catalogo = "grupo06proyectofinal"
esquema = "sismos"
volumen = f"/Volumes/{catalogo}/{esquema}"

spark.sql(f"USE CATALOG {catalogo}")
spark.sql(f"USE SCHEMA {esquema}")
spark.conf.set("spark.sql.session.timeZone", "America/Lima")

## 3. Funciones: escritura, multicolinealidad y selección univariada

In [ ]:
def a_spark(pdf, esquema=None):
    pdf = pdf.copy()
    for c in pdf.columns:
        if isinstance(pdf[c].dtype, (pd.StringDtype, pd.ArrowDtype)):
            pdf[c] = pdf[c].astype(object)
        if pdf[c].dtype == object:
            pdf[c] = pdf[c].map(lambda v: v if v is None or isinstance(v, (str, bytes, list, dict, np.ndarray))
                                else (None if pd.isna(v) else str(v)))
    return spark.createDataFrame(pdf, esquema) if esquema is not None else spark.createDataFrame(pdf)


def escribir(df, tabla, particion=None, modo="overwrite", contar=True):
    escritor = df.write.format("delta").mode(modo)
    if modo == "overwrite":
        escritor = escritor.option("overwriteSchema", "true")
    if particion:
        escritor = escritor.partitionBy(particion)
    escritor.saveAsTable(tabla)
    if contar:
        n = spark.table(tabla).count()
        print(f"  {tabla}: {n:,} filas")
        return n


def subir_pandas(lotes, tabla, esquema=None, preparar=None, particion=None, mb_lote=64):
    if isinstance(lotes, pd.DataFrame):
        lotes = [lotes]
    modo, n_lotes = "overwrite", 0
    for pdf in lotes:
        if len(pdf) == 0:
            continue
        partes = max(1, int(np.ceil(pdf.memory_usage(deep=True).sum() / 1e6 / mb_lote)))
        for idx in np.array_split(np.arange(len(pdf)), partes):
            sdf = a_spark(pdf.iloc[idx], esquema)
            if preparar is not None:
                sdf = preparar(sdf)
            escribir(sdf, tabla, particion, modo, contar=False)
            modo, n_lotes = "append", n_lotes + 1
    n = spark.table(tabla).count()
    print(f"  {tabla}: {n:,} filas ({n_lotes} lotes)")
    return n


def inventario(prefijo):
    filas = []
    for t in spark.sql("SHOW TABLES").collect():
        if t.isTemporary or not t.tableName.startswith(prefijo):
            continue
        d = spark.sql(f"DESCRIBE DETAIL {t.tableName}").first()
        filas.append({"tabla": t.tableName, "filas": spark.table(t.tableName).count(), "mb": round((d["sizeInBytes"] or 0) / 1e6, 1)})
    return pd.DataFrame(filas)

In [ ]:
UMBRAL_CORRELACION = 0.90
P_VALOR = 0.05
HOLDOUT_PCT = 30
seleccion_filas = []


def multicolinealidad(df, columnas):
    corr = Correlation.corr(VectorAssembler(inputCols=columnas, outputCol="num_vec").transform(df).select("num_vec"),
                            "num_vec", "pearson").head()[0].toArray()
    descartar = set()
    for i in range(len(columnas)):
        for j in range(i + 1, len(columnas)):
            if abs(corr[i, j]) > UMBRAL_CORRELACION:
                descartar.add(columnas[j])
    print("Drop por multicolinealidad:", sorted(descartar))
    return [c for c in columnas if c not in descartar]


def seleccion_univariada(df, columnas, target, tipo_feature, tipo_label):
    selector = UnivariateFeatureSelector(featuresCol="vec", labelCol=target, selectionMode="fpr")
    selector.setFeatureType(tipo_feature).setLabelType(tipo_label).setSelectionThreshold(P_VALOR)
    modelo = selector.fit(VectorAssembler(inputCols=columnas, outputCol="vec").transform(df).select("vec", target))
    elegidas = [columnas[i] for i in modelo.selectedFeatures]
    print(f"Seleccionadas (p < {P_VALOR}):", elegidas, "| descartadas:", [c for c in columnas if c not in elegidas])
    return elegidas

## 4. Regresión: variables derivadas y holdout

In [ ]:
def perfil_e030(vs30):
    return (F.when(vs30 > 1500, "S0").when(vs30 > 500, "S1").when(vs30 >= 180, "S2").otherwise("S3"))


target_reg = "ln_amp"
esp = (spark.table("gold_registros").where("ciudad = 'Lima' AND vs30 IS NOT NULL AND dist_hipo_km IS NOT NULL")
       .select("evento", "archivo_estacion", "estacion", "lat", "lon", "vs30", "dist_hipo_km", "periodos_s",
               F.posexplode("sa_geo").alias("indice_periodo", "Sa"))
       .where("Sa > 0")
       .withColumn("T", F.element_at("periodos_s", F.col("indice_periodo") + 1))
       .drop("periodos_s")
       .withColumn("ln_sa_r", F.log(F.col("Sa") * F.col("dist_hipo_km")))
       .withColumn("ln_T", F.log("T"))
       .withColumn("ln_vs30", F.log("vs30"))
       .withColumn("perfil_e030", perfil_e030(F.col("vs30"))))

eventos = sorted(r[0] for r in esp.select("evento").distinct().collect())
evento_holdout = eventos[-1]
cubeta = F.abs(F.hash(F.col("evento"), F.col("estacion"))) % 100
esp = esp.withColumn("es_holdout", (F.col("evento") == evento_holdout) & (cubeta < HOLDOUT_PCT))
print("Sismos:", eventos, "| holdout: 30 % de las estaciones de", evento_holdout)

## 5. Regresión: término del sismo y columnas que no entran al modelo
Sa, ln(Sa·R), el término del sismo, las coordenadas y la distancia contienen el target o lo definen: entrar con ellas sería una fuga de información.

In [ ]:
eta = esp.where(~F.col("es_holdout")).groupBy("evento", "indice_periodo").agg(F.avg("ln_sa_r").alias("eta"))
fe_reg = esp.join(eta, ["evento", "indice_periodo"]).withColumn(target_reg, F.col("ln_sa_r") - F.col("eta"))

cols_id_reg = ["evento", "archivo_estacion", "estacion", "indice_periodo", "es_holdout"]
cols_fuga_reg = ["Sa", "ln_sa_r", "eta", "lat", "lon", "dist_hipo_km"]
df_sel_reg = fe_reg.where(F.col("evento") < evento_holdout)

print("Correlación de Pearson con el target (columnas que NO entran al modelo):")
for c in cols_fuga_reg:
    print(f"  {c:15s} {df_sel_reg.stat.corr(target_reg, c):.3f}")

## 6. Regresión: multicolinealidad y F-regression

In [ ]:
cols_categoricas_reg = [c.name for c in fe_reg.schema if c.dataType == StringType() and c.name not in cols_id_reg + cols_fuga_reg]
cols_nums_reg = [c.name for c in fe_reg.schema if c.dataType != StringType() and c.name not in cols_id_reg + cols_fuga_reg + [target_reg]]
cols_nums_reg = sorted(cols_nums_reg, key=lambda c: not c.startswith("ln_"))
print("Numericas:", cols_nums_reg, "| Categoricas:", cols_categoricas_reg)

num_filtradas_reg = multicolinealidad(df_sel_reg, cols_nums_reg)
seleccionadas_f = seleccion_univariada(df_sel_reg, num_filtradas_reg, target_reg, "continuous", "continuous")
if "ln_T" not in seleccionadas_f:
    seleccionadas_f.insert(0, "ln_T")

## 7. Regresión: Chi-cuadrado para la variable categórica
El target se discretiza en 5 quintiles (`QuantileDiscretizer`) para poder usar Chi-cuadrado con el perfil de suelo de la E.030.

In [ ]:
df_chi = df_sel_reg
for c in cols_categoricas_reg:
    df_chi = StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep").fit(df_chi).transform(df_chi)
df_chi = QuantileDiscretizer(numBuckets=5, inputCol=target_reg, outputCol="target_bin").fit(df_chi).transform(df_chi)
elegidas_idx = seleccion_univariada(df_chi, [f"{c}_idx" for c in cols_categoricas_reg], "target_bin", "categorical", "categorical")
seleccionadas_chi = [c for c in cols_categoricas_reg if f"{c}_idx" in elegidas_idx]
print("Categoricas seleccionadas:", seleccionadas_chi)

In [ ]:
seleccion_filas += [("regresion", c, "num", evento_holdout) for c in seleccionadas_f]
seleccion_filas += [("regresion", c, "cat", evento_holdout) for c in seleccionadas_chi]

escribir(fe_reg, "fe_regresion", particion="evento")

## 8. Clasificación (onda P): ventanas de STEAD y holdout

In [ ]:
target_clf = "etiqueta"
LOTE_STEAD = "stead_chunk2"
cubeta_p = F.abs(F.hash(F.col("trace_name"))) % 100
fe_clf = (spark.table("gold_onda_p_ventanas")
          .withColumn(target_clf, F.col(target_clf).cast("double"))
          .withColumn("lote", F.lit(LOTE_STEAD))
          .withColumn("es_holdout", cubeta_p < HOLDOUT_PCT))
cols_id_clf = ["trace_name", "lote", "es_holdout"]
cols_fuga_clf = ["tipo"]
fe_clf.groupBy("tipo").agg(F.count("*").alias("ventanas"), F.avg(target_clf).alias("etiqueta_media")).show()

cols_nums_clf = [c.name for c in fe_clf.schema if c.dataType != StringType() and c.name not in cols_id_clf + cols_fuga_clf + [target_clf]]
df_sel_clf = fe_clf.where(~F.col("es_holdout"))
print("Numericas:", len(cols_nums_clf), cols_nums_clf)

## 9. Clasificación: multicolinealidad y ANOVA

In [ ]:
num_filtradas_clf = multicolinealidad(df_sel_clf, cols_nums_clf)
seleccionadas_anova = seleccion_univariada(df_sel_clf, num_filtradas_clf, target_clf, "continuous", "categorical")

seleccion_filas += [("clasificacion", c, "num", LOTE_STEAD) for c in seleccionadas_anova]
escribir(fe_clf, "fe_clasificacion")

## 10. Tablas de control de ML
Registro de modelos, monitoreo y métricas. Se crean una sola vez.

In [ ]:
spark.sql("""CREATE TABLE IF NOT EXISTS fe_columnas_seleccionadas (
    caso STRING, columna STRING, tipo STRING, lote STRING) PARTITIONED BY (caso)""")

spark.sql("""CREATE TABLE IF NOT EXISTS ml_registro_modelos (
    caso STRING, version INT, algoritmo STRING, lote STRING, ruta STRING, promovido BOOLEAN,
    columnas_num STRING, columnas_cat STRING, snapshots STRING, metricas STRING, detalle STRING)""")

spark.sql("""CREATE TABLE IF NOT EXISTS ml_monitoreo (
    clave STRING, lote STRING, caso STRING, champion_version INT, n_features_drift INT,
    metrica_base DOUBLE, metrica_nueva DOUBLE, degradacion BOOLEAN, reentrenar BOOLEAN, motivo STRING)
    PARTITIONED BY (caso)""")

spark.sql("""CREATE TABLE IF NOT EXISTS ml_metricas (
    modelo STRING, metrica STRING, valor DOUBLE, parametros STRING, fecha STRING, notebook STRING)
    PARTITIONED BY (notebook)""")
spark.sql("CREATE VOLUME IF NOT EXISTS modelos")

## 11. Variables seleccionadas

In [ ]:
seleccion = spark.createDataFrame(seleccion_filas, "caso string, columna string, tipo string, lote string")
seleccion.write.mode("overwrite").option("replaceWhere", "caso IN ('regresion', 'clasificacion')").saveAsTable("fe_columnas_seleccionadas")
display(spark.table("fe_columnas_seleccionadas").groupBy("caso").agg(F.count("*").alias("n_variables")).toPandas())